# WithGyro Experiment 0.1 — 60-channel temporal representation probe

## Research question

Does adding the 30 angular-acceleration wavelet event channels to the existing 30 linear-acceleration event channels improve linearly accessible gesture information under fixed-duration and relative-progress temporal aggregation?

Input to every representation is **only the first 60 event channels** of the derived 66-channel Angular66 dataset:

- channels `0:30`: linear-acceleration polarity-split wavelet events;
- channels `30:60`: angular-acceleration polarity-split wavelet events;
- channels `60:66`: raw acceleration + gyroscope, explicitly excluded from the probe.

The experiment follows the user-disjoint direct-event probing protocol used by Experiment 1.3.1 / 1.3.1.2. Computation is performed by the Slurm CPU array; this notebook is analysis-only.

## Protocol

- dataset: `outputs/action0_wavelets_0e5_1_2_4_8_sr_64_with_gyro_0e5_1_2_4_8/low-pass/aligned-board-events`;
- 60 unsigned event channels at 64 Hz;
- labels: `A B C D E X G H I J K L`;
- split seeds: `(11, 23, 37, 53, 71)`; 12/4/4 train/validation/test users;
- fixed-duration sweep: `50 ms, 150 ms` (start 50 ms, +100 ms, stop at 200 ms);
- relative-progress sweep: `1, 2, 4, 6, 8, 10, 12, 16, 20` bins;
- feature value: channel-wise weighted event **count/sum** in each bin;
- train-only per-feature z-score standardization;
- classifiers: multinomial Logistic Regression (`linear`) and 5-NN;
- primary metric: Balanced Accuracy; Accuracy and Macro-F1 are also saved.

At 64 Hz the requested fixed durations quantize to 3 samples = 46.875 ms and 10 samples = 156.25 ms.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')


REPO_ROOT = find_repo_root()
RESULTS_DIR = (
    REPO_ROOT / 'notebooks/artifacts/withGyro/experiment_0_1_temporal_representation_probe/linear_angular_accel_60event_v1'
)
RESULTS_PATH = RESULTS_DIR / 'experiment_0_1_results.csv'
SUMMARY_PATH = RESULTS_DIR / 'experiment_0_1_summary.csv'
PROVENANCE_PATH = RESULTS_DIR / 'provenance.json'

for path in (RESULTS_PATH, SUMMARY_PATH, PROVENANCE_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            f'Missing finalized artifact: {path}\n'
            'Run: bash scripts/bash_script/withGyro/submit_exp_0_1_pipeline.bash'
        )

results = pd.read_csv(RESULTS_PATH)
summary = pd.read_csv(SUMMARY_PATH)
provenance = json.loads(PROVENANCE_PATH.read_text(encoding='utf-8'))
print('Repository root:', REPO_ROOT)
print('Results:', RESULTS_PATH)
print('Finalized rows:', len(results))
print('Expected runs:', provenance['expected_runs'])

## 1. Final protocol and provenance

In [ ]:
display(pd.Series(provenance, name='value').to_frame())

## 2. Aggregate test results

In [ ]:
display(
    summary[[
        'representation_family', 'condition', 'requested_duration_ms',
        'actual_duration_ms', 'n_bins', 'feature_dim', 'classifier',
        'n_splits', 'mean_test_balanced_accuracy', 'sd_test_balanced_accuracy',
        'mean_test_accuracy', 'mean_test_macro_f1',
    ]]
)

## 3. Fixed-duration sweep

In [ ]:
fixed = summary[summary.representation_family == 'fixed_duration'].copy()
fig, ax = plt.subplots(figsize=(8.5, 5.0))
for classifier, group in fixed.groupby('classifier'):
    group = group.sort_values('actual_duration_ms')
    ax.errorbar(
        group.actual_duration_ms,
        group.mean_test_balanced_accuracy,
        yerr=group.sd_test_balanced_accuracy,
        marker='o',
        capsize=4,
        label=classifier,
    )
ax.set_xlabel('Actual fixed-bin duration (ms)')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('WithGyro Exp 0.1 — Fixed-duration event aggregation')
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 4. Relative-progress sweep

In [ ]:
relative = summary[summary.representation_family == 'relative_progress'].copy()
fig, ax = plt.subplots(figsize=(8.5, 5.0))
for classifier, group in relative.groupby('classifier'):
    group = group.sort_values('n_bins')
    ax.errorbar(
        group.n_bins,
        group.mean_test_balanced_accuracy,
        yerr=group.sd_test_balanced_accuracy,
        marker='o',
        capsize=4,
        label=classifier,
    )
ax.set_xlabel('Number of relative-progress bins')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('WithGyro Exp 0.1 — Relative-progress event aggregation')
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 5. Best conditions

In [ ]:
best = (
    summary.sort_values('mean_test_balanced_accuracy', ascending=False)
    .groupby(['representation_family', 'classifier'], as_index=False)
    .head(1)
    [[
        'representation_family', 'classifier', 'condition', 'n_bins',
        'actual_duration_ms', 'feature_dim', 'mean_test_balanced_accuracy',
        'sd_test_balanced_accuracy', 'mean_test_accuracy', 'mean_test_macro_f1',
    ]]
)
display(best)

## Interpretation

Use the Linear curves as the main measure of how much class information is directly linearly accessible after adding angular-acceleration events. The 5-NN curves are a secondary local-geometry probe. Fixed-duration aggregation preserves physical time and is compatible with causal streaming bins; relative-progress aggregation normalizes gesture phase but requires the final gesture duration, so it remains an offline endpoint-dependent representation.

For a clean gyro contribution claim, compare these finalized 60-event results against the matching 30-event Experiment 1.3.1 conditions using the same split seeds and representation definitions.